# Support Vector Machines: RBF Kernel and Hyperparameter Tuning
Accompanies **Unit 4.3: Support Vector Machines (SVM)**.

A soft-margin Support Vector Machine maximizes the geometric margin width $\frac{2}{\|\mathbf{w}\|_2}$ while penalizing slack margin violations $\xi_i$:
$$\min_{\mathbf{w}, b, \boldsymbol{\xi}} \frac{1}{2} \|\mathbf{w}\|_2^2 + C \sum_{i=1}^n \xi_i$$

For non-linear boundaries, the **RBF (Gaussian) Kernel** maps inputs into an implicit infinite-dimensional Hilbert space:
$$K(\mathbf{x}_i, \mathbf{x}_j) = \exp(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|_2^2)$$

Key Hyperparameters to Tune:
- **$C$ (Slack Penalty)**:
  - Small $C$: Tolerates large margin violations (wide margin, high bias, prevents overfitting).
  - Large $C$: Penalizes violations heavily (narrow margin, low bias, risk of overfitting).
- **$\gamma$ (Kernel Bandwidth $\frac{1}{2\sigma^2}$)**:
  - Small $\gamma$: Broad Gaussian bell curve. Each point has far-reaching influence (smooth boundary).
  - Large $\gamma$: Narrow Gaussian bell curve. Boundary wraps tightly around individual support vector islands (severe overfitting).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_circles
from sklearn.model_selection import GridSearchCV

X, y = make_circles(n_samples=400, noise=0.08, factor=0.4, random_state=42)

### Grid Search Across 2D Logarithmic Space ($C$ and $\gamma$)

In [ ]:
param_grid = {
    'C': [0.1, 1.0, 10.0, 100.0],
    'gamma': [0.01, 0.1, 1.0, 10.0]
}

grid_svm = GridSearchCV(SVC(kernel='rbf'), param_grid=param_grid, cv=5)
grid_svm.fit(X, y)

print(f"Optimal Hyperparameters: {grid_svm.best_params_}")
print(f"Best CV Accuracy:        {grid_svm.best_score_:.4f}")

### Visualizing the Decision Boundary and Support Vectors

In [ ]:
best_model = grid_svm.best_estimator_

x_min, x_max = X[:, 0].min() - 0.4, X[:, 0].max() + 0.4
y_min, y_max = X[:, 1].min() - 0.4, X[:, 1].max() + 0.4
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
Z = best_model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7, 6))
plt.contourf(xx, yy, Z, levels=[-10, 0, 10], alpha=0.25, colors=['crimson', 'dodgerblue'])
plt.contour(xx, yy, Z, levels=[-1, 0, 1], linestyles=['--', '-', '--'], colors='black', linewidths=[1, 2, 1])

plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='k', s=35)
sv = best_model.support_vectors_
plt.scatter(sv[:, 0], sv[:, 1], s=110, facecolors='none', edgecolors='gold', linewidths=1.8, label=f'Support Vectors ({len(sv)} pts)')

plt.title(f'Tuned RBF SVM (C={best_model.C}, gamma={best_model.gamma})', fontsize=12, fontweight='bold')
plt.legend(loc='upper right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()